In [0]:
import time

from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType


# ==============================================================================
# Helper: Safe table existence check for Unity Catalog 3-level namespace
# ==============================================================================

def check_table_exists(session, full_table_name: str) -> bool:
    """
    Safely verifies if a 3-level namespace table exists in Unity Catalog.
    Returns True if table exists, False if it does not.
    """
    try:
        return session.catalog.tableExists(full_table_name)
    except Exception:
        return False


# ==============================================================================
# Audit logging
# ==============================================================================

def log_audit_metrics(
    session,
    audit_table: str,
    target_table: str,
    batch_id: int,
    input_count: int,
    written_count: int,
    rejected_count: int,
    duration_sec: float,
    status: str,
    error_message: str = None
):
    """Write batch-level metrics to the audit Delta table."""

    audit_data = [(
        target_table,
        int(batch_id),
        int(input_count),
        int(written_count),
        int(rejected_count),
        float(duration_sec),
        status,
        str(error_message)[:4000] if error_message else None
    )]

    schema = """
        target_table STRING,
        batch_id LONG,
        input_count LONG,
        written_count LONG,
        rejected_count LONG,
        duration_sec DOUBLE,
        status STRING,
        error_message STRING
    """

    audit_df = (
        session.createDataFrame(audit_data, schema=schema)
        .withColumn("logged_at", F.current_timestamp())
    )

    (
        audit_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(audit_table)
    )


# ==============================================================================
# Micro-batch processing
# ==============================================================================

def process_micro_batch(
    batch_df,
    batch_id: int,
    target_table: str,
    audit_table: str
):
    """Enrich and write a micro-batch, then record ingestion metrics."""

    session = batch_df.sparkSession
    start_time = time.time()

    input_count = 0
    written_count = 0

    try:
        input_count = batch_df.count()

        if input_count == 0:
            return

        # Ensure the rescued-data column exists.
        if "_rescued_data" not in batch_df.columns:
            batch_df = batch_df.withColumn(
                "_rescued_data",
                F.lit(None).cast("string")
            )

        # Include source fields and rescued content in the hash.
        # Exclude the helper column containing the physical file path.
        payload_cols = sorted([
            name for name in batch_df.columns
            if name != "_bronze_file_path"
        ])

        # JSON serialization supports nested structs and arrays.
        # Compute the hash before adding ingestion metadata.
        enriched_df = batch_df.withColumn(
            "record_hash",
            F.sha2(
                F.to_json(
                    F.struct(*[
                        F.col(f"`{name}`")
                        for name in payload_cols
                    ]),
                    options={"ignoreNullFields": "false"}
                ),
                256
            )
        )

        # Preserve the batch identifier supplied by the source CSV.
        if "batch_id" in enriched_df.columns:
            enriched_df = enriched_df.withColumnRenamed(
                "batch_id",
                "source_batch_id"
            )

        enriched_df = (
            enriched_df
            .withColumn(
                "ingestion_timestamp",
                F.current_timestamp()
            )
            .withColumn(
                "source_filename",
                F.col("_bronze_file_path")
            )
            .withColumn(
                "batch_id",
                F.lit(batch_id).cast("long")
            )
            .withColumn(
                "load_status",
                F.when(
                    F.col("_rescued_data").isNotNull(),
                    F.lit("RESCUED")
                ).otherwise(F.lit("SUCCESS"))
            )
            .drop("_bronze_file_path")
        )

        rescued_count = enriched_df.filter(
            F.col("load_status") == "RESCUED"
        ).count()

        # ----------------------------------------------------------------------
        # CHECK IF TABLE EXISTS: CREATE FROM SCRATCH vs APPEND / MERGE
        # ----------------------------------------------------------------------
        table_exists = check_table_exists(session, target_table)

        if not table_exists:
            print(f"Table {target_table} does not exist. Creating from scratch...")
            (
                enriched_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(target_table)
            )
            written_count = input_count

        else:
            print(f"Table {target_table} exists. Merging / appending new records...")
            enriched_df.createOrReplaceTempView("staged_micro_batch")

            session.sql(f"""
                MERGE INTO {target_table} AS target
                USING staged_micro_batch AS source
                ON target.record_hash = source.record_hash
                WHEN NOT MATCHED THEN INSERT *
            """)

            # Read the actual number of inserted rows.
            last_commit = (
                session.sql(f"DESCRIBE HISTORY {target_table}")
                .orderBy(F.desc("version"))
                .select("operationMetrics")
                .first()
            )

            metrics = last_commit["operationMetrics"] or {}
            written_count = int(metrics.get("numTargetRowsInserted", 0))

        duration_sec = round(time.time() - start_time, 2)
        rejected_count = 0

        log_audit_metrics(
            session=session,
            audit_table=audit_table,
            target_table=target_table,
            batch_id=batch_id,
            input_count=input_count,
            written_count=written_count,
            rejected_count=rejected_count,
            duration_sec=duration_sec,
            status=(
                "SUCCESS_WITH_RESCUED"
                if rescued_count > 0
                else "SUCCESS"
            )
        )

        print(
            f"[{target_table}] Batch {batch_id}: "
            f"input={input_count}, "
            f"inserted={written_count}, "
            f"skipped={input_count - written_count}, "
            f"rescued_in_input={rescued_count}, "
            f"duration={duration_sec}s"
        )

    except Exception as err:
        duration_sec = round(time.time() - start_time, 2)

        try:
            log_audit_metrics(
                session=session,
                audit_table=audit_table,
                target_table=target_table,
                batch_id=batch_id,
                input_count=input_count,
                written_count=written_count,
                rejected_count=0,
                duration_sec=duration_sec,
                status="FAILED",
                error_message=str(err)
            )
        except Exception as audit_err:
            print(f"Audit logging also failed: {audit_err}")

        raise


# ==============================================================================
# Ingestion function
# ==============================================================================

def ingest_to_bronze(
    source_path: str,
    file_format: str,
    target_table: str,
    checkpoint_path: str,
    audit_table: str
):
    """Load CSV files or JSON arrays into a Bronze Delta table."""

    file_format = file_format.lower()

    if file_format == "json":
        json_schema = (
            spark.read
            .option("multiLine", "true")
            .option("primitivesAsString", "true")
            .option("mode", "FAILFAST")
            .option("pathGlobFilter", "*.json")
            .json(source_path)
            .schema
        )

        required_columns = {
            "event_id",
            "equipment_id",
            "sensor",
            "reading"
        }

        missing_columns = (
            required_columns - set(json_schema.fieldNames())
        )

        if missing_columns:
            raise ValueError(
                f"Missing JSON fields: {sorted(missing_columns)}"
            )

        json_files = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "binaryFile")
            .option("pathGlobFilter", "*.json")
            .load(source_path)
        )

        raw_df = (
            json_files
            .select(
                F.col("path").alias("_bronze_file_path"),
                F.explode(
                    F.from_json(
                        F.decode(F.col("content"), "UTF-8"),
                        ArrayType(json_schema),
                        options={"mode": "FAILFAST"}
                    )
                ).alias("event")
            )
            .select(
                "event.*",
                "_bronze_file_path"
            )
            .withColumn(
                "_rescued_data",
                F.lit(None).cast("string")
            )
        )

    else:
        read_options = {
            "cloudFiles.format": file_format,
            "cloudFiles.schemaLocation": f"{checkpoint_path}/schema",
            "cloudFiles.schemaEvolutionMode": "addNewColumns",
            "rescuedDataColumn": "_rescued_data",
            "cloudFiles.inferColumnTypes": "true"
        }

        if file_format == "csv":
            read_options["header"] = "true"

        raw_df = (
            spark.readStream
            .format("cloudFiles")
            .options(**read_options)
            .load(source_path)
            .select(
                "*",
                F.col("_metadata.file_path").alias(
                    "_bronze_file_path"
                )
            )
        )

    query = (
        raw_df.writeStream
        .option(
            "checkpointLocation",
            f"{checkpoint_path}/data"
        )
        .trigger(availableNow=True)
        .foreachBatch(
            lambda df, b_id: process_micro_batch(
                df,
                b_id,
                target_table,
                audit_table
            )
        )
        .start()
    )

    query.awaitTermination()

    print(f"Ingestion completed for table: {target_table}\n")


# ==============================================================================
# Pipeline configuration
# ==============================================================================

CATALOG_NAME = "data2026acc123456"
SCHEMA_NAME = "team3"

BASE_SOURCE_PATH = (
    f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/datasets"
)

BASE_CHECKPOINT_PATH = (
    f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/checkpoints"
)

AUDIT_LOG_TABLE = (
    f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_ingestion_audit"
)

spark.conf.set("spark.sql.session.timeZone", "UTC")


# ==============================================================================
# Run all four sources sequentially
# ==============================================================================

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/equipment",
    file_format="csv",
    target_table=f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_equipment",
    checkpoint_path=f"{BASE_CHECKPOINT_PATH}/equipment",
    audit_table=AUDIT_LOG_TABLE
)

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/maintenance_history",
    file_format="csv",
    target_table=(
        f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_maintenance_history"
    ),
    checkpoint_path=f"{BASE_CHECKPOINT_PATH}/maintenance_history",
    audit_table=AUDIT_LOG_TABLE
)

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/work_orders",
    file_format="csv",
    target_table=f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_work_orders",
    checkpoint_path=f"{BASE_CHECKPOINT_PATH}/work_orders",
    audit_table=AUDIT_LOG_TABLE
)

ingest_to_bronze(
    source_path=f"{BASE_SOURCE_PATH}/sensor_events",
    file_format="json",
    target_table=f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_sensor_events",
    checkpoint_path=(
        f"{BASE_CHECKPOINT_PATH}/sensor_events_binary_v1"
    ),
    audit_table=AUDIT_LOG_TABLE
)

print("All Bronze ingestions finished and audited.")

Ingestion completed for table: data2026acc123456.team3.bronze_equipment

Ingestion completed for table: data2026acc123456.team3.bronze_maintenance_history

Ingestion completed for table: data2026acc123456.team3.bronze_work_orders

Ingestion completed for table: data2026acc123456.team3.bronze_sensor_events

All Bronze ingestions finished and audited.
